# Expense Tracker — Initial ML Model Training (Kaggle)

This notebook is the canonical Kaggle entry point for the initial ML training run.

It bootstraps the exact repository branch, installs Python 3.14 with uv, fetches fresh data, trains the complete master pipeline, evaluates the selected category classifier, and packages the complete model run.

**Enable a Kaggle GPU accelerator before running the training cell.**

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Yoge-2004/expense-tracker.git"
REPO_REF = "feature/ml-expense-intelligence"
WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / "expense-tracker"
ML_DIR = REPO_DIR / "ml"
RUNS_DIR = WORK_ROOT / "expense_ml_initial_runs"
CACHE_ROOT = Path("/kaggle/temp/huggingface")
RUNS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_ROOT)
os.environ["HF_DATASETS_CACHE"] = str(CACHE_ROOT / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_ROOT / "transformers")
print("Branch:", REPO_REF)
print("Working directory:", WORK_ROOT)

In [ ]:
def run(*args, cwd=None, env=None, check=True):
    merged_env = os.environ.copy()
    if env:
        merged_env.update({key: str(value) for key, value in env.items()})
    print("$", " ".join(map(str, args)))
    return subprocess.run(args, cwd=cwd, env=merged_env, check=check)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run("git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR))
run("git", "rev-parse", "HEAD", cwd=REPO_DIR)

## 1. Verify the Kaggle GPU

In [ ]:
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU detected. Enable a Kaggle GPU accelerator before training.")
run("nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader")

## 2. Install Python 3.14 and the project environment

In [ ]:
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = shutil.which("uv") or str(Path.home() / ".local/bin/uv")
if not Path(UV).exists():
    raise FileNotFoundError("uv was installed but could not be located")
run(UV, "python", "install", "3.14")
run(UV, "sync", "--all-extras", "--dev", cwd=ML_DIR)
run(UV, "run", "python", "--version", cwd=ML_DIR)
run(
    UV, "run", "python", "-c",
    "import torch; print('Torch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'); print('BF16:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)",
    cwd=ML_DIR,
)

## 3. Load Hugging Face credentials from Kaggle Secrets

Create a Kaggle Secret named `HF_TOKEN` or `HUGGINGFACE_TOKEN`.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            value = secrets.get_secret(secret_name)
        except Exception:
            value = None
        if value:
            os.environ["HF_TOKEN"] = value
            break
except Exception as exc:
    print("Kaggle Secrets API unavailable:", exc)

print("HF_TOKEN loaded." if os.environ.get("HF_TOKEN") else "HF_TOKEN not found; gated datasets may fail.")

## 4. Start from a fresh dataset cache

This deliberately removes prepared data and source caches so the initial model cannot silently reuse an older normalization/taxonomy result.

In [ ]:
data_dir = ML_DIR / "data"
for path in (data_dir / "prepared", data_dir / "cache", data_dir / "raw"):
    if path.exists():
        print("Removing:", path)
        shutil.rmtree(path)

print("Fresh data workspace ready.")

## 5. Kaggle training resources

In [ ]:
cpu_count = os.cpu_count() or 4
KAGGLE_ENV = {
    "EXPENSE_ML_CPU_THREADS": min(cpu_count, 8),
    "EXPENSE_ML_TORCH_THREADS": min(cpu_count, 8),
    "EXPENSE_ML_DATALOADER_WORKERS": min(max(cpu_count // 2, 1), 4),
    "EXPENSE_ML_BATCH_SIZE": 16,
    "EXPENSE_ML_EVAL_BATCH_SIZE": 64,
    "EXPENSE_ML_MIXED_PRECISION": "auto",
}
os.environ.update({key: str(value) for key, value in KAGGLE_ENV.items()})
print(json.dumps(KAGGLE_ENV, indent=2))

## 6. Prepare all configured datasets

In [ ]:
run(
    UV, "run", "expense-ml", "prepare",
    "--config", "config/datasets.yaml",
    "--output", "data/prepared/transactions.parquet",
    cwd=ML_DIR, env=KAGGLE_ENV,
)
prepared = ML_DIR / "data/prepared/transactions.parquet"
if not prepared.exists():
    raise FileNotFoundError(f"Prepared dataset was not created: {prepared}")
run(
    UV, "run", "python", "-c",
    "import pyarrow.parquet as pq; print('Prepared rows:', pq.ParquetFile('data/prepared/transactions.parquet').metadata.num_rows)",
    cwd=ML_DIR,
)

## 7. Train the complete initial ML pipeline

This cell intentionally does **not** use `--no-transformer`. The run should train TF-IDF, XLM-R, merchant similarity, duplicate similarity, anomaly detection when applicable, and spending forecasting when applicable.

In [ ]:
run(
    UV, "run", "python", "-m", "expense_ml.master_pipeline",
    "--config", "config/datasets.yaml",
    "--prepared", "data/prepared/transactions.parquet",
    "--output", str(RUNS_DIR),
    cwd=ML_DIR, env=KAGGLE_ENV,
)

## 8. Inspect the model-selection and final test results

In [ ]:
run_dirs = sorted(path for path in RUNS_DIR.iterdir() if path.is_dir())
if not run_dirs:
    raise FileNotFoundError("No training run directory was produced.")
RUN_DIR = run_dirs[-1]
MANIFEST_PATH = RUN_DIR / "manifest.json"
MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
print("Run:", RUN_DIR.name)
print("Status:", MANIFEST.get("status"))
print("Selected model:", MANIFEST.get("selected_model"))
print("Test evaluation:", MANIFEST.get("test_evaluation"))
print("India holdout:", MANIFEST.get("india_holdout_evaluation"))
print("\nModel statuses:")
for name, details in MANIFEST.get("models", {}).items():
    print(f"  {name}: {details.get('status')}")

In [ ]:
for relative in (
    "reports/model_selection_validation.json",
    "reports/model_comparison.json",
    "reports/category_test.json",
    "reports/category_test_country_metrics.json",
    "reports/category_india_holdout.json",
):
    path = RUN_DIR / relative
    print(f"\n--- {relative} ---")
    if path.exists():
        print(path.read_text(encoding="utf-8")[:16000])
    else:
        print("not present")

## 9. Package the complete initial model run

In [ ]:
archive_base = WORK_ROOT / f"expense-tracker-ml-initial-run-{RUN_DIR.name}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR))
print("Canonical Kaggle artifact:", archive_path)
print("Size (MiB):", round(archive_path.stat().st_size / 1024**2, 2))

## 10. Optional publication to Hugging Face

Keep this disabled until the metrics have been inspected. When enabled, only the selected category model is promoted to the versioned and `production` revisions.

In [ ]:
PUBLISH_TO_HF = False
HF_MODEL_REPO = os.environ.get("HF_MODEL_REPO", "Yoge-2004/expense-intelligence-model")
HF_MODEL_PRIVATE = os.environ.get("HF_MODEL_PRIVATE", "1") == "1"

if PUBLISH_TO_HF:
    token = os.environ.get("HF_TOKEN", "")
    if not token:
        raise RuntimeError("HF_TOKEN is required for publication.")
    selected = MANIFEST["selected_model"]["name"]
    selected_dir = RUN_DIR / "models" / ("category-transformer" if selected == "transformer" else "category-tfidf")
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    api.create_repo(repo_id=HF_MODEL_REPO, repo_type="model", private=HF_MODEL_PRIVATE, exist_ok=True)
    revision = f"v{RUN_DIR.name}"
    for target in (revision, "production"):
        api.create_branch(repo_id=HF_MODEL_REPO, repo_type="model", branch=target, revision="main", exist_ok=True)
        api.upload_folder(repo_id=HF_MODEL_REPO, repo_type="model", folder_path=str(selected_dir), revision=target, commit_message=f"Publish Expense Tracker initial model {revision}")
        api.upload_file(path_or_fileobj=str(MANIFEST_PATH), path_in_repo="manifest.json", repo_id=HF_MODEL_REPO, repo_type="model", revision=target, commit_message="Publish initial training manifest")
    print({"repo_id": HF_MODEL_REPO, "revision": revision, "production_revision": "production", "selected_model": selected})
else:
    print("Hugging Face publication skipped.")